In [1]:
import pandas as pd #used for data manipulation.
import numpy as np #used for mathematical functions.
import matplotlib.pyplot as plt #used for charts.
import matplotlib
%matplotlib inline
import seaborn as sns #used for complex charts.

matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.dpi'] = 150

from IPython.core.pylabtools import figsize

In [2]:
df1 = pd.read_csv('dataco_preprocessed.csv', encoding='latin1')

In [4]:
# Prep: Select mining features + encode

In [6]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Select high-value mining columns
cols = [
    "Order Region",
    "Shipping Mode",
    "Order Item Total",
    "Order Profit Per Order",
    "Late_delivery_risk"
]

data = df1[cols].copy()

In [7]:
# Encode categorical variables
le = LabelEncoder()
for col in ["Order Region", "Shipping Mode", "Late_delivery_risk"]:
    data[col] = le.fit_transform(data[col])

# Scale numeric features
scaler = StandardScaler()
num_cols = ["Order Item Total", "Order Profit Per Order"]
data[num_cols] = scaler.fit_transform(data[num_cols])

In [8]:
# Classification: Predict Late Delivery Risk

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X = data.drop("Late_delivery_risk", axis=1)
y = data["Late_delivery_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.21      0.28      0.24      6505
           1       0.20      0.17      0.19      6416
           2       0.27      0.32      0.29     10150
           3       0.80      0.50      0.62     18306
           4       0.21      0.34      0.26      8627
           5       0.20      0.10      0.14      2123
           6       0.22      0.13      0.16      2029

    accuracy                           0.35     54156
   macro avg       0.30      0.26      0.27     54156
weighted avg       0.42      0.35      0.37     54156



In [10]:
import pandas as pd

importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importance)

Shipping Mode             0.950357
Order Profit Per Order    0.022102
Order Item Total          0.018632
Order Region              0.008909
dtype: float64


In [11]:
# Association Rule Mining (Hidden Operational Rules)

In [23]:
%pip install mlxtend

  Obtaining dependency information for mlxtend from https://files.pythonhosted.org/packages/a9/f9/f62b6c3bd08e1269d9bca9aabe0d990a7c3ffbfbcc495135ad0b59ce860b/mlxtend-0.24.0-py3-none-any.whl.metadata
  Using cached mlxtend-0.24.0-py3-none-any.whl.metadata (7.3 kB)
Using cached mlxtend-0.24.0-py3-none-any.whl (1.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [24]:
pip list

Package                       VersionNote: you may need to restart the kernel to use updated packages.

----------------------------- -------------------
absl-py                       2.1.0
aext_assistant                0.4.0
aext_assistant_server         0.4.0
aext_core                     0.4.0
aext_core_server              0.4.0
aext_shared                   0.4.0
aiobotocore                   2.23.2
aiodns                        3.5.0
aiohappyeyeballs              2.6.1
aiohttp                       3.12.15
aioitertools                  0.12.0
aiosignal                     1.4.0
alabaster                     0.7.16
altair                        5.2.0
anaconda-anon-usage           0.4.2
anaconda-auth                 0.8.6
anaconda-catalogs             0.2.0
anaconda-cli-base             0.5.2
anaconda-client               1.13.0
anaconda-navigator            2.6.6
anaconda-project              0.11.1
annotated-types               0.6.0
anyio                         3.5.0
appdirs    

In [25]:
from mlxtend.frequent_patterns import apriori, association_rules

# Discretize for rule mining
rules_df = df1[
    ["Order Region", "Shipping Mode", "Late_delivery_risk"]
].astype(str)

# One-hot encode
encoded = pd.get_dummies(rules_df)

# Mine frequent itemsets
frequent = apriori(
    encoded,
    min_support=0.05,
    use_colnames=True
)

rules = association_rules(
    frequent,
    metric="confidence",
    min_threshold=0.6
)

# Focus on late delivery rules
late_rules = rules[
    rules["consequents"].astype(str)
    .str.contains("Late_delivery_risk_1")
]

late_rules.sort_values("lift", ascending=False).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski


In [26]:
# Clustering: Order Typologies

In [29]:
from sklearn.cluster import KMeans

cluster_features = data[
    ["Order Item Total", "Order Profit Per Order"]
]

kmeans = KMeans(n_clusters=4, random_state=42)
data["Cluster"] = kmeans.fit_predict(cluster_features)

# Cluster profiles
cluster_summary = (
    df1.assign(Cluster=data["Cluster"])
    .groupby("Cluster")[[
        "Order Item Total",
        "Order Profit Per Order",
        "Late_delivery_risk"
    ]]
    .mean()
)

cluster_summary

,Order Item Total,Order Profit Per Order,Late_delivery_risk
Cluster,,,
0,0.105704,0.254221,0.426982
1,0.190144,0.819073,0.426031
2,0.044978,-0.130231,0.428229
3,0.135965,-3.414975,0.431832


In [30]:
# Anomaly Detection: Profit & Cost Outliers

In [31]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination=0.02,
    random_state=42
)

data["Anomaly"] = iso.fit_predict(
    data[["Order Item Total", "Order Profit Per Order"]]
)

anomalies = df1[data["Anomaly"] == -1]
anomalies.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Late_delivery_risk,Order Country,Order Item Discount,Order Item Quantity,Order Item Total,Order Profit Per Order,Order Region,Product Category Id,Product Price,Shipping Mode,Order Profit Per Order_log,reward
34,PAYMENT,-0.922361,0.25,0.500000,China,6.56,1,0.162334,-4.285561,Eastern Asia,73,327.750000,First Class,NaN,0.744910
124,TRANSFER,0.925251,1.00,0.500000,Venezuela,20.00,5,0.244481,1.512976,South America,9,99.989998,Standard Class,5.198387,0.853053
125,TRANSFER,0.309380,1.00,0.333333,PanamÃ¡,25.00,5,0.241894,-7.714450,Central America,9,99.989998,Standard Class,NaN,0.673361
126,TRANSFER,1.541122,1.00,0.666667,PanamÃ¡,25.00,5,0.241894,1.699698,Central America,9,99.989998,Standard Class,5.300714,0.852659
127,TRANSFER,1.541122,1.00,0.666667,Guatemala,50.00,5,0.228962,-2.795518,Central America,9,99.989998,Standard Class,NaN,0.763117


In [32]:
# RL-Ready State–Action Mining

In [33]:
state_action = (
    df1.groupby(["Order Region", "Shipping Mode"])
    .agg(
        avg_profit=("Order Profit Per Order", "mean"),
        late_rate=("Late_delivery_risk", "mean"),
        order_count=("Order Profit Per Order", "count")
    )
    .reset_index()
)

state_action.sort_values("avg_profit", ascending=False).head()

,Order Region,Shipping Mode,avg_profit,late_rate,order_count
2,Canada,Second Class,0.141876,0.671687,166
9,Central Africa,Same Day,0.091305,0.383966,79
49,South America,Same Day,0.084402,0.423540,776
66,Southern Africa,Second Class,0.081754,0.643333,150
20,East Africa,First Class,0.076952,0.500000,280
